# Prioritizing Candidate Genetic Variants Using GTEx, HuBMAP, and IDG


## What is this module about?

This module begins with a [whole-genome sequencing study of early-onset
advanced heart failure](https://doi.org/10.1038/s41598-025-88465-8)
([PMID: 39910139](https://pubmed.ncbi.nlm.nih.gov/39910139/)). The researchers
studied 102 Swedish heart transplant or heart-assist-device recipients. They
used a computer pipeline to rank variants and clinical experts to classify
them. Their Supplementary Table S4 reports 54 variant rows from 46 people
across 25 genes.

You will use that complete published table. At each step, you will add a
different kind of context to its 25 genes: tissue expression from GTEx,
cell-type expression from HuBMAP, and protein information from Pharos. These
resources can guide research follow-up. They do not replace the study's variant
classification or prove that a variant causes disease.

## Learning objectives

By the end of this module, you will be able to:

1. Use GTEx to check whether genes linked to candidate variants are expressed
   in a disease-related tissue.
2. Use HuBMAP to study gene expression in specific cell types while accounting
   for missing data and small groups.
3. Use IDG and Pharos to interpret protein information and druggability.
4. Combine several data sources to select candidate variants for more study.
5. Explain what an application programming interface, or API, is and how APIs
   support repeatable research.

## Data sources used in this module

| Resource | What it is | How this module uses it |
|---|---|---|
| [Candidate variant table](data/variants.csv) and [source paper](https://doi.org/10.1038/s41598-025-88465-8) | The local table contains all 54 rows from the paper's Supplementary Table S4. | Provides the published HGVS names, study scores, classifications, phenotypes, and comments. |
| [GTEx v10](https://gtexportal.org/home/) | GTEx is an NIH Common Fund reference resource for studying gene expression and genetic regulation across human tissues. | A GTEx Portal REST API response provides median expression in heart atrial appendage and left ventricle. |
| [HuBMAP](https://portal.hubmapconsortium.org/) | HuBMAP is an NIH Common Fund program that maps cells and molecules within human tissues. | A HuBMAP Cells API response provides expression summaries for selected heart cell types. |
| [IDG](https://commonfund.nih.gov/IDG/) and [Pharos](https://pharos.nih.gov/) | IDG is an NIH Common Fund program focused on understudied druggable proteins. Pharos is its integrated protein-target information resource. | A Pharos GraphQL API response provides protein information and target development levels. |
| [Integrated teaching table](data/integrated_prioritization.csv) | A local table created by joining the published variants with the saved GTEx, HuBMAP, and Pharos data by gene symbol. | Returns gene-level context to all 54 variant rows while keeping each evidence layer separate. |

The activities query GTEx, HuBMAP, and Pharos by default. Saved API responses
are included as a backup if a live service is unavailable.

## How do I use this module?

Work through the pages in order. Plan for about two hours.

- **Activities:** Run the same analysis code in the website or in the matching
  notebook.
- **Knowledge checks:** Select an answer after each main activity.
- **Complete workflow:** Use the integrated notebook to run every evidence
  layer in one place.
- **Live APIs:** The activities request current data by default. Each resource
  page shows how to switch to the saved response if needed.


# Introduction to APIs


## What is an API?

API stands for **application programming interface**. An API gives software a
standard way to ask another service for data or an action.

In this module, code takes the 25 genes from the heart-failure paper and sends
their identifiers to biological data services. The services return information
in a format that Python can read. This is easier to repeat than copying results
from a website by hand.

## Parts of an API request

| Term | Meaning |
|---|---|
| Endpoint | The web address used for a specific task |
| Request | The message sent to the API |
| Parameter | A value that controls the request, such as a gene or tissue ID |
| Response | The information returned by the API |
| JSON | A common text format used to organize returned data |

A reproducible API request records the endpoint, parameters, resource version,
and date.

## APIs used in this module

| CFDE resource | How its API is used |
|---|---|
| [GTEx Portal API](https://gtexportal.org/api/v2/docs) | Request median expression for named genes and heart tissues |
| [HuBMAP APIs](https://docs.hubmapconsortium.org/apis.html) | Find selected heart cells and retrieve indexed expression values |
| [Pharos GraphQL API](https://pharos.nih.gov/api) | Request selected protein-target fields, including target development level |

The published variant table is saved in the repository. GTEx, HuBMAP, and
Pharos are queried live by default, with dated saved responses as backups.
ClinVar was one input to the paper's original ranking pipeline, but this module
does not query ClinVar or repeat the paper's clinical classification.

## REST and GraphQL

GTEx and HuBMAP provide REST-style APIs. A REST API usually has several
endpoints. Each endpoint handles a task, such as retrieving expression data or
finding cells. Parameters tell the endpoint which genes, tissues, or cells to
return.

**GraphQL** is an API query language. A GraphQL service often uses one endpoint.
The request contains a query that names the exact fields needed. Variables hold
values that can change, such as a gene symbol. The response follows the shape
of the query, so the client receives the fields it requested.

Pharos uses GraphQL. This query asks for a protein symbol, name, UniProt ID,
target development level, and publication count:

```graphql
query TargetContext($symbol: String!) {
  target(q: {sym: $symbol}) {
    sym
    name
    uniprot
    tdl
    publicationCount
  }
}
```

GraphQL controls which fields are returned. It does not make those fields
evidence that a variant causes disease.

## Other common biomedical APIs

| API | Example use |
|---|---|
| [NCBI E-utilities](https://www.ncbi.nlm.nih.gov/home/develop/api/) | Search and retrieve records from NCBI resources such as Gene, PubMed, and Protein |
| [Ensembl REST API](https://rest.ensembl.org/) | Retrieve genes, variants, sequences, and comparative genomics data |
| [UCSC Genome Browser API](https://genome.ucsc.edu/goldenpath/help/api.html) | Retrieve genome sequences and selected annotation tracks |

APIs work best for focused requests. Complete downloads are often better for
large datasets.

## Build an API Request Using E-utilities

This activity uses NCBI E-utilities as a simple example of an API URL. It is not
part of the module's candidate variant workflow and does not retrieve ClinVar
records.

The code builds an NCBI Gene search URL without sending the request. Change
`MYH7` to another gene symbol and choose **Run Code**.


In [ ]:
from urllib.parse import urlencode

endpoint = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
parameters = {
    "db": "gene",
    "term": "MYH7[gene] AND Homo sapiens[organism]",
    "retmode": "json",
}
request_url = f"{endpoint}?{urlencode(parameters)}"
request_url
